# 06 — Evaluasi Sistem NIDS pada VPS Uji (simulasi serangan terkendali)

**Tujuan:** mengukur kinerja sistem NIDS yang berjalan *real-time* di VPS uji terhadap serangan yang **waktunya
diketahui pasti**, karena dijalankan sendiri dari laptop peneliti dan dicatat otomatis di `jadwal_serangan.csv`.

**Urutan kerja sebelum menjalankan notebook ini:**
1. Baseline baru VPS uji (dengan halaman web) selesai direkam: 5 Nov 15.29 – 8 Nov 15.29 WITA.
2. Model VPS uji dilatih ulang dari baseline itu dan dipasang di server.
3. Skenario dijalankan dari laptop dengan `04_Kode\5_simulasi_serangan\simulasi.py` (lihat README folder itu).
4. Log deteksi disalin ke laptop: `python ambil_log.py`.

**Pelabelan** (lihat `evaluasi.py`):
- *serangan*: aliran dari IP laptop selama jendela skenario serangan (+130 detik, karena aliran baru dianggap selesai
  paling lambat 120 detik setelah paket terakhirnya);
- *normal*: aliran dari IP laptop selama skenario "normal" (pengunjung web);
- *latar*: semua aliran lain (bot internet). Dianggap normal, karena model dilatih dari lalu lintas semacam ini.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

FOLDER = os.path.abspath('')                                              # 04_Kode/4_notebook
SIM = os.path.abspath(os.path.join(FOLDER, '..', '5_simulasi_serangan'))
DATA = os.path.abspath(os.path.join(FOLDER, '..', '..', '05_Data_Eksperimen', 'Server-Uji'))
sys.path.insert(0, SIM)
import evaluasi

JADWAL = os.path.join(DATA, 'jadwal_serangan.csv')
LOG = os.path.join(DATA, 'hasil', 'deteksi_vps_uji.db')
jadwal = evaluasi.muat_jadwal(JADWAL)
dari = jadwal.mulai_us.min() - 3600e6                                     # 1 jam sebelum skenario pertama
sampai = jadwal.selesai_us.max() + 3600e6                                 # 1 jam setelah skenario terakhir
aliran, anomali, notif, model = evaluasi.muat_log(LOG, dari, sampai)
print('Skenario: %d | aliran pada periode evaluasi: %s | anomali: %s' % (len(jadwal), f'{len(aliran):,}', f'{len(anomali):,}'))
print('Model aktif:', ', '.join(model.versi))
jadwal[['id', 'skenario', 'label', 'mulai_wita', 'selesai_wita', 'ip_sumber', 'jumlah', 'berhasil', 'parameter']]

## 1. Metrik keseluruhan (per aliran)

In [ ]:
df = evaluasi.beri_label(aliran, jadwal)
print(df.label.value_counts().to_string(), '\n')
hasil = pd.DataFrame({
    'Negatif = normal + latar': evaluasi.metrik(df, ('normal', 'latar')),
    'Negatif = normal (pengunjung web) saja': evaluasi.metrik(df, ('normal',)),
}).T
hasil.to_csv(os.path.join(DATA, 'hasil', 'metrik_keseluruhan.csv'))
hasil.style.format({'precision': '{:.3f}', 'recall': '{:.3f}', 'F1': '{:.3f}', 'FPR': '{:.4f}'})

## 2. Hasil per skenario

- **tertangkap (%)**: persentase aliran skenario yang ditandai anomali.
- **terdeteksi**: minimal satu aliran ditandai (administrator mendapat peringatan).
- **deteksi pertama**: waktu dari mulainya serangan sampai anomali pertama dicatat di log.
- **latensi notifikasi**: median selisih waktu anomali dicatat di log sampai Telegram mengonfirmasi pesan.

In [ ]:
ps = evaluasi.per_skenario(df, anomali, notif, jadwal)
ps.to_csv(os.path.join(DATA, 'hasil', 'hasil_per_skenario.csv'), index=False)
ps

## 3. Garis waktu: anomali per menit dan jendela skenario

In [ ]:
wita = lambda s: pd.to_datetime(s, unit='us', utc=True).dt.tz_convert('Asia/Makassar')
menit = df.assign(t=wita(df.waktu_selesai_us).dt.floor('min')).groupby('t').agg(aliran=('id', 'size'), anomali=('ditandai', 'sum'))
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(menit.index, menit.anomali, width=1 / 1440, color='#b42318', label='aliran ditandai anomali')
for _, j in jadwal.iterrows():
    a, b = wita(pd.Series([j.mulai_us]))[0], wita(pd.Series([j.selesai_us]))[0]
    ax.axvspan(a, b, color='#1f6feb' if j.label == 'serangan' else '#1a7f37', alpha=0.15)
    ax.text(a, ax.get_ylim()[1] * 0.9, j.skenario, fontsize=8)
ax.set_ylabel('anomali / menit'); ax.set_xlabel('waktu (WITA)'); ax.legend(loc='upper right')
ax.set_title('Biru = jendela serangan, hijau = pengunjung normal')
plt.tight_layout(); plt.savefig(os.path.join(DATA, 'hasil', 'garis_waktu_evaluasi.png'), dpi=150); plt.show()

## 4. Alarm di luar serangan (alarm palsu) — untuk dibahas di BAB IV

In [ ]:
fp = df[df.ditandai & (df.label != 'serangan')]
print('Aliran bukan serangan yang ditandai: %d' % len(fp))
(fp.groupby(['label', 'src_ip']).agg(aliran=('id', 'size'), port=('dst_port', lambda s: ', '.join(map(str, s.value_counts().index[:4]))),
                                     model=('oleh', lambda s: ','.join(sorted(set(s)))))
   .sort_values('aliran', ascending=False).head(20))